# Evaluación Formativa 1 — Código del informe
## Dataset: *E-Commerce Shipping Data* (entregas atrasadas)

**MCDI501 - Estadística Computacional para la Toma de Decisiones**
Magíster en Ciencia de Datos e Inteligencia Artificial, Universidad Andrés Bello
Equipo: José Ignacio Rodríguez, Yerko Gallardo y Sebastián Rojas

Este cuaderno reproduce los resultados del informe y sigue su mismo orden:

0. Configuración y carga de datos
1. Estadística descriptiva (ID1.2): calidad de datos y descriptiva
2. Estimación puntual e intervalos de confianza (ID1.3)
3. Pruebas de hipótesis (ID1.4)
4. Síntesis

## 0. Configuración y carga de datos

El archivo original `data/raw/ecommerce_shipping.csv` no se modifica. La base de trabajo `df` solo cambia
los nombres de las columnas y agrega `entrega_atrasada` (1 = atrasada, 0 = a tiempo). No se eliminan filas.

In [ ]:
from pathlib import Path
import pandas as pd

# Raíz del proyecto: funciona al ejecutar desde la raíz o desde notebooks/
RAIZ = Path.cwd().resolve()
while not (RAIZ / 'pyproject.toml').exists():
    RAIZ = RAIZ.parent

raw = pd.read_csv(RAIZ / 'data/raw/ecommerce_shipping.csv', encoding='utf-8-sig')

NOMBRES = {
    'ID': 'id_envio', 'Warehouse_block': 'bloque_bodega', 'Mode_of_Shipment': 'modalidad_envio',
    'Customer_care_calls': 'llamadas_atencion', 'Customer_rating': 'calificacion_cliente',
    'Cost_of_the_Product': 'costo_producto', 'Prior_purchases': 'compras_previas',
    'Product_importance': 'importancia_producto', 'Gender': 'genero',
    'Discount_offered': 'descuento_ofrecido', 'Weight_in_gms': 'peso_gramos',
}
TARGET = 'entrega_atrasada'

df = raw.rename(columns=NOMBRES)
df[TARGET] = raw['Reached.on.Time_Y.N']
df['importancia_producto'] = pd.Categorical(df['importancia_producto'],
                                            categories=['low', 'medium', 'high'], ordered=True)

print('Dimensiones:', df.shape)
df.head()

## 1. Estadística descriptiva (ID1.2)

### 1.1 Calidad de datos
Nulos, duplicados, unicidad del identificador y tasa de atraso.

In [ ]:
n = len(df)
sin_id = [c for c in raw.columns if c != 'ID']

print(f"Registros                  : {n:,}")
print(f"Celdas vacías              : {raw.isna().sum().sum()}")
print(f"ID únicos                  : {df['id_envio'].nunique():,}")
print(f"Duplicados con ID          : {raw.duplicated().sum()}")
print(f"Duplicados sin ID          : {raw.duplicated(subset=sin_id).sum()}")
print(f"Entregas atrasadas         : {df[TARGET].sum():,} ({df[TARGET].mean()*100:.1f}%)")
print(f"Entregas a tiempo          : {(df[TARGET] == 0).sum():,} ({(1 - df[TARGET].mean())*100:.1f}%)")
print('Bodegas:', df['bloque_bodega'].value_counts().sort_index().to_dict())

### 1.2 Patrones a vigilar
Los umbrales se observaron mirando el resultado: son exploratorios, no pruebas de hipótesis.

In [ ]:
desc_alto = df['descuento_ofrecido'] > 10
peso_medio = df['peso_gramos'].between(2001, 4000)

for nombre, grupo in [('Descuento > 10', desc_alto), ('Peso entre 2.001 y 4.000 g', peso_medio)]:
    print(f"{nombre:28s}: n={grupo.sum():,}  atraso={df.loc[grupo, TARGET].mean()*100:.1f}%")
    print(f"{'Resto':28s}: n={(~grupo).sum():,}  atraso={df.loc[~grupo, TARGET].mean()*100:.1f}%")

### 1.3 Tabla de controles
Resumen de los controles, exportado a `outputs/tablas/calidad_datos.csv`. Estados: OK, ALERTA (revisar) e
INFO (hallazgo, no es un error).

In [ ]:
filas = []
def control(nombre, variable, esperado, observado, ok, detalle=''):
    filas.append((nombre, variable, esperado, observado, 'OK' if ok else 'ALERTA', detalle))

control('filas', 'todas', n, len(df), len(df) == n, 'Sin exclusiones de filas')
control('nulos_totales', 'todas', 0, int(raw.isna().sum().sum()), raw.isna().sum().sum() == 0, 'Sin imputación ni faltantes artificiales')
control('id_unico', 'id_envio', n, int(df['id_envio'].nunique()), df['id_envio'].is_unique, 'No prueba independencia ni un registro por cliente')

dominios = {'bloque_bodega': ['A', 'B', 'C', 'D', 'F'], 'modalidad_envio': ['Flight', 'Road', 'Ship'], 'genero': ['F', 'M']}
for v, esperado in dominios.items():
    obs = sorted(df[v].unique())
    control('dominio_categorico', v, esperado, obs, obs == esperado, 'F no se recodifica a E' if v == 'bloque_bodega' else '')
    control('suma_frecuencias', v, n, int(df[v].value_counts().sum()), df[v].value_counts().sum() == n)

obs = sorted(df['importancia_producto'].dropna().astype(str).unique())
control('dominio_ordinal', 'importancia_producto', 'low<medium<high', obs, obs == ['high', 'low', 'medium'])
control('dominio_ordinal', 'calificacion_cliente', '1..5', f"{df['calificacion_cliente'].min()}..{df['calificacion_cliente'].max()}", df['calificacion_cliente'].between(1, 5).all())
control('resultado_binario', TARGET, '{0,1}', sorted(df[TARGET].unique().tolist()), set(df[TARGET]) <= {0, 1},
        f"1 (atrasada)={df[TARGET].sum()}; 0 (a tiempo)={(df[TARGET] == 0).sum()}")
control('alias_igual_original', TARGET, 0, int((df[TARGET] != raw['Reached.on.Time_Y.N']).sum()), (df[TARGET] == raw['Reached.on.Time_Y.N']).all())

numericas = ['llamadas_atencion', 'costo_producto', 'compras_previas', 'descuento_ofrecido', 'peso_gramos']
for v in numericas:
    s = df[v]
    q1, q3 = s.quantile([0.25, 0.75])
    bajo, alto = int((s < q1 - 1.5 * (q3 - q1)).sum()), int((s > q3 + 1.5 * (q3 - q1)).sum())
    control('rango_valido', v, '>0', f'{s.min()}..{s.max()}', (s > 0).all())
    filas.append(('extremos_regla_IQR', v, 'conservar', f'bajo={bajo}; alto={alto}', 'INFO' if bajo + alto else 'OK',
                  'Valores válidos; no se eliminan' if bajo + alto else ''))

control('duplicados_con_ID', 'todas', 0, int(raw.duplicated().sum()), not raw.duplicated().any())
control('duplicados_sin_ID', 'todas menos ID', 0, int(raw.duplicated(subset=sin_id).sum()), not raw.duplicated(subset=sin_id).any(),
        'Perfiles idénticos no se eliminan automáticamente')

for nombre, var, etiqueta, grupo, detalle in [
        ('hallazgo_descuento', 'descuento_ofrecido', 'descuento>10', desc_alto, 'Exploratorio. Posible variable posterior al resultado; unidad no especificada'),
        ('hallazgo_peso', 'peso_gramos', 'peso 2001-4000 g', peso_medio, 'Exploratorio. Distribución con hueco entre 2.000 y 4.000 g')]:
    obs = (f"{etiqueta}: n={grupo.sum()}, atraso={df.loc[grupo, TARGET].mean():.3f}; "
           f"resto: n={(~grupo).sum()}, atraso={df.loc[~grupo, TARGET].mean():.3f}")
    filas.append((nombre, var, '-', obs, 'INFO', detalle))

calidad = pd.DataFrame(filas, columns=['control', 'variable', 'esperado', 'observado', 'estado', 'detalle'])
(RAIZ / 'outputs/tablas').mkdir(parents=True, exist_ok=True)
calidad.to_csv(RAIZ / 'outputs/tablas/calidad_datos.csv', index=False, encoding='utf-8')

print(calidad['estado'].value_counts().to_dict())
calidad[['control', 'variable', 'observado', 'estado']]